In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_PATH = "sla_prep_outputs/prepared_stage1_dataset.csv"
OUTPUT_DIR = Path("cohort_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_DAYS = 60
MIN_COHORT_SIZE = 10


In [ ]:
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    datetime_cols = [
        "lead_created_at",
        "sale_ts",
        "lead_Дата перехода в Сборку",
        "handed_to_delivery_ts",
        "lead_Дата перехода Передан в доставку",
        "issued_or_pvz_ts",
        "received_ts",
        "rejected_ts",
        "returned_ts",
        "closed_ts",
    ]
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if "sale_date" in df.columns:
        df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce").dt.date

    if "buyout_flag" in df.columns:
        vals = (
            df["buyout_flag"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({"<na>": np.nan, "nan": np.nan, "": np.nan})
        )
        df["buyout_flag"] = vals.map({"true": True, "false": False, "1": True, "0": False})

    if "outcome_unknown" in df.columns:
        vals = (
            df["outcome_unknown"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({"<na>": np.nan, "nan": np.nan, "": np.nan})
        )
        df["outcome_unknown"] = vals.map({"true": True, "false": False, "1": True, "0": False})

    return df


df = load_data(INPUT_PATH)
print("Размер входного датасета:", df.shape)
display(df.head(3))


In [ ]:
def prepare_cohort_base(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    stats = {"rows_input": int(len(df))}

    df = df[df["sale_date"].notna()].copy()
    stats["rows_with_sale_date"] = int(len(df))

    received = df["received_ts"].notna() if "received_ts" in df.columns else pd.Series(False, index=df.index)
    rejected = df["rejected_ts"].notna() if "rejected_ts" in df.columns else pd.Series(False, index=df.index)
    returned = df["returned_ts"].notna() if "returned_ts" in df.columns else pd.Series(False, index=df.index)
    buyout_flag = df["buyout_flag"].fillna(False) if "buyout_flag" in df.columns else pd.Series(False, index=df.index)
    outcome_unknown = df["outcome_unknown"].fillna(False) if "outcome_unknown" in df.columns else pd.Series(False, index=df.index)

    df["is_buyout"] = received | buyout_flag
    df["is_non_buyout"] = rejected | returned

    known_outcome_mask = (~outcome_unknown) & (df["is_buyout"] | df["is_non_buyout"])
    stats["excluded_unknown_or_open"] = int((~known_outcome_mask).sum())
    df = df.loc[known_outcome_mask].copy()

    non_buyout_ts = pd.concat(
        [
            df["rejected_ts"] if "rejected_ts" in df.columns else pd.Series(pd.NaT, index=df.index),
            df["returned_ts"] if "returned_ts" in df.columns else pd.Series(pd.NaT, index=df.index),
        ],
        axis=1
    ).min(axis=1)

    df["outcome_ts"] = np.where(df["is_buyout"], df["received_ts"], non_buyout_ts)
    df["outcome_ts"] = pd.to_datetime(df["outcome_ts"], errors="coerce")

    sale_date_ts = pd.to_datetime(df["sale_date"], errors="coerce")
    df["days_to_outcome"] = (df["outcome_ts"].dt.normalize() - sale_date_ts.dt.normalize()).dt.days

    df["days_to_buyout"] = np.where(
        df["is_buyout"],
        (df["received_ts"].dt.normalize() - sale_date_ts.dt.normalize()).dt.days,
        np.nan
    )

    negative_mask = df["days_to_outcome"].notna() & (df["days_to_outcome"] < 0)
    stats["excluded_negative_days"] = int(negative_mask.sum())
    df = df.loc[~negative_mask].copy()

    too_long_mask = df["days_to_outcome"].notna() & (df["days_to_outcome"] > MAX_DAYS)
    stats["rows_outcome_after_horizon"] = int(too_long_mask.sum())

    stats["rows_final"] = int(len(df))
    stats["buyout_orders"] = int(df["is_buyout"].sum())
    stats["non_buyout_orders"] = int(df["is_non_buyout"].sum())

    return df, stats


cohort_base, cohort_stats = prepare_cohort_base(df)
print(json.dumps(cohort_stats, ensure_ascii=False, indent=2, default=str))
display(cohort_base.head(5))


In [ ]:
def build_cohort_matrix(df: pd.DataFrame, max_days: int = 60):
    cohort_sizes = (
        df.groupby("sale_date")["lead_id"]
        .nunique()
        .rename("cohort_size")
        .reset_index()
    )

    buyouts = df[df["is_buyout"]].copy()
    buyouts = buyouts[buyouts["days_to_buyout"].notna()].copy()
    buyouts["days_to_buyout"] = buyouts["days_to_buyout"].astype(int)

    records = []
    for cohort_date, group in df.groupby("sale_date"):
        cohort_size = group["lead_id"].nunique()
        buyout_days = buyouts.loc[buyouts["sale_date"] == cohort_date, "days_to_buyout"].tolist()

        for day in range(max_days + 1):
            cumulative_buyouts = sum(d <= day for d in buyout_days)
            records.append(
                {
                    "sale_date": cohort_date,
                    "day_number": day,
                    "cohort_size": cohort_size,
                    "cumulative_buyouts": cumulative_buyouts,
                    "cumulative_buyout_rate": cumulative_buyouts / cohort_size if cohort_size else np.nan,
                }
            )

    cohort_long = pd.DataFrame(records)

    cohort_matrix = cohort_long.pivot(
        index="sale_date",
        columns="day_number",
        values="cumulative_buyout_rate"
    ).sort_index()

    return cohort_sizes, cohort_long, cohort_matrix


cohort_sizes, cohort_long, cohort_matrix = build_cohort_matrix(cohort_base, MAX_DAYS)

print("Размер таблицы cohort_long:", cohort_long.shape)
print("Размер матрицы cohort_matrix:", cohort_matrix.shape)

display(cohort_sizes.head())
display(cohort_long.head())
display(cohort_matrix.head())


In [ ]:
def build_cohort_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby("sale_date")
        .agg(
            cohort_size=("lead_id", "nunique"),
            buyout_orders=("is_buyout", "sum"),
            non_buyout_orders=("is_non_buyout", "sum"),
        )
        .reset_index()
    )

    summary["final_buyout_rate"] = summary["buyout_orders"] / summary["cohort_size"]

    buyout_stats = (
        df[df["is_buyout"]]
        .groupby("sale_date")["days_to_buyout"]
        .agg(avg_days_to_buyout="mean", median_days_to_buyout="median")
        .reset_index()
    )

    summary = summary.merge(buyout_stats, on="sale_date", how="left")
    return summary.sort_values("sale_date")


cohort_summary = build_cohort_summary(cohort_base)
display(cohort_summary.head(10))


In [ ]:
plt.figure(figsize=(16, 10))
data = cohort_matrix.to_numpy(dtype=float)
plt.imshow(data, aspect="auto", interpolation="nearest")
plt.colorbar(label="Накопительный % выкупа")

plt.xlabel("День с момента продажи")
plt.ylabel("Когорта (дата продажи)")
plt.title("Когортный heatmap накопительного выкупа")

yticks = np.arange(len(cohort_matrix.index))
ylabels = [str(x) for x in cohort_matrix.index]
step = max(1, len(yticks) // 20)
plt.yticks(yticks[::step], ylabels[::step])

xticks = np.arange(0, cohort_matrix.shape[1], max(1, cohort_matrix.shape[1] // 12))
plt.xticks(xticks, xticks)

plt.tight_layout()
plt.show()


In [ ]:
eligible = cohort_sizes[cohort_sizes["cohort_size"] >= MIN_COHORT_SIZE].copy()
eligible = eligible.sort_values("sale_date")

plt.figure(figsize=(14, 8))
if not eligible.empty:
    idx = np.linspace(0, len(eligible) - 1, min(8, len(eligible)), dtype=int)
    selected_dates = eligible.iloc[idx]["sale_date"].tolist()

    for cohort_date in selected_dates:
        temp = cohort_long[cohort_long["sale_date"] == cohort_date]
        plt.plot(
            temp["day_number"],
            temp["cumulative_buyout_rate"] * 100,
            label=str(cohort_date)
        )

plt.xlabel("День с момента продажи")
plt.ylabel("Накопительный выкуп, %")
plt.title("Рост выкупа по дням для выбранных когорт")
plt.legend(title="Когорта", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 7))
plt.hist(cohort_summary["final_buyout_rate"].dropna() * 100, bins=20)
plt.xlabel("Итоговый выкуп по когорте, %")
plt.ylabel("Количество когорт")
plt.title("Распределение итогового процента выкупа по когортам")
plt.tight_layout()
plt.show()


In [ ]:
cohort_base.to_csv(OUTPUT_DIR / "cohort_base_dataset.csv", index=False)
cohort_sizes.to_csv(OUTPUT_DIR / "cohort_sizes.csv", index=False)
cohort_long.to_csv(OUTPUT_DIR / "cohort_growth_long.csv", index=False)
cohort_matrix.to_csv(OUTPUT_DIR / "cohort_growth_matrix.csv")
cohort_summary.to_csv(OUTPUT_DIR / "cohort_summary.csv", index=False)
(OUTPUT_DIR / "cohort_stats.json").write_text(
    json.dumps(cohort_stats, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

# heatmap
plt.figure(figsize=(16, 10))
data = cohort_matrix.to_numpy(dtype=float)
plt.imshow(data, aspect="auto", interpolation="nearest")
plt.colorbar(label="Накопительный % выкупа")
plt.xlabel("День с момента продажи")
plt.ylabel("Когорта (дата продажи)")
plt.title("Когортный heatmap накопительного выкупа")
yticks = np.arange(len(cohort_matrix.index))
ylabels = [str(x) for x in cohort_matrix.index]
step = max(1, len(yticks) // 20)
plt.yticks(yticks[::step], ylabels[::step])
xticks = np.arange(0, cohort_matrix.shape[1], max(1, cohort_matrix.shape[1] // 12))
plt.xticks(xticks, xticks)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cohort_heatmap.png", dpi=200, bbox_inches="tight")
plt.close()

# selected cohort lines
plt.figure(figsize=(14, 8))
if not eligible.empty:
    for cohort_date in selected_dates:
        temp = cohort_long[cohort_long["sale_date"] == cohort_date]
        plt.plot(
            temp["day_number"],
            temp["cumulative_buyout_rate"] * 100,
            label=str(cohort_date)
        )
plt.xlabel("День с момента продажи")
plt.ylabel("Накопительный выкуп, %")
plt.title("Рост выкупа по дням для выбранных когорт")
plt.legend(title="Когорта", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cohort_lines_selected.png", dpi=200, bbox_inches="tight")
plt.close()

# final distribution
plt.figure(figsize=(12, 7))
plt.hist(cohort_summary["final_buyout_rate"].dropna() * 100, bins=20)
plt.xlabel("Итоговый выкуп по когорте, %")
plt.ylabel("Количество когорт")
plt.title("Распределение итогового процента выкупа по когортам")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cohort_final_buyout_distribution.png", dpi=200, bbox_inches="tight")
plt.close()

print("Файлы сохранены в:", OUTPUT_DIR.resolve())
